# Phase 1 — Duplicate Audit and Reproducible Data Partitioning

## Objective

This notebook completes the Phase 1 data pipeline by auditing duplicate records and creating deterministic train, validation, and test partitions.

The analysis covers:

- Exact duplicate records
- Repeated pre-treatment feature vectors
- Duplicate-handling limitations
- Deterministic 60/20/20 data partitioning
- Partition-level treatment and outcome validation
- Cross-partition feature-leakage checks

## Key Limitation

The public dataset does not contain a user identifier. Therefore, identical rows cannot automatically be interpreted as duplicated users. Different users may share the same anonymized features, treatment assignment, and outcomes.

Duplicate records are audited and documented but are not removed.

## Step 1 — Set Up the Analysis

In [1]:
from pathlib import Path
import time

import duckdb
import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start_path: Path) -> Path:
    """Find the project root containing the data and src folders."""
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "criteo-uplift-v2.1.parquet"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "criteo-uplift-v2.1-partitioned.parquet"
TEMP_OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "criteo-uplift-v2.1-partitioned.tmp.parquet"

EXPECTED_ROW_COUNT = 13_979_592
PARTITION_SEED = "adlift-v1"
feature_columns = [f"f{i}" for i in range(12)]
binary_columns = ["treatment", "conversion", "visit", "exposure"]
all_columns = [*feature_columns, *binary_columns]

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

INPUT_SQL_PATH = str(INPUT_FILE).replace("'", "''")
OUTPUT_SQL_PATH = str(OUTPUT_FILE).replace("'", "''")
TEMP_OUTPUT_SQL_PATH = str(TEMP_OUTPUT_FILE).replace("'", "''")

feature_sql = ", ".join(feature_columns)
feature_string_sql = ", ".join(
    f"CAST({feature} AS VARCHAR)" for feature in feature_columns
)

partition_expression = f"""
CASE
    WHEN md5_number_lower(
        concat_ws('|', '{PARTITION_SEED}', {feature_string_sql})
    ) % 100 < 60 THEN 'train'
    WHEN md5_number_lower(
        concat_ws('|', '{PARTITION_SEED}', {feature_string_sql})
    ) % 100 < 80 THEN 'validation'
    ELSE 'test'
END
"""

connection = duckdb.connect()
connection.execute("SET preserve_insertion_order = true")

print(f"Input dataset: {INPUT_FILE}")
print(f"Partitioned dataset: {OUTPUT_FILE}")
print(f"Partition seed: {PARTITION_SEED}")
print("Analysis environment ready.")

Input dataset: /Users/ever/Library/CloudStorage/OneDrive2-ColumbiaUniversityIrvingMedicalCenter/adlift项目/data/processed/criteo-uplift-v2.1.parquet
Partitioned dataset: /Users/ever/Library/CloudStorage/OneDrive2-ColumbiaUniversityIrvingMedicalCenter/adlift项目/data/processed/criteo-uplift-v2.1-partitioned.parquet
Partition seed: adlift-v1
Analysis environment ready.


## Step 2 — Audit Exact Duplicate Records

An exact duplicate group contains rows with identical values across all 16 published fields. Because no user identifier is available, this is a record-level audit rather than a user-level duplicate test.

In [ ]:
duplicate_summary = connection.execute(
    f"""
    WITH duplicate_groups AS (
        SELECT
            *,
            COUNT(*) AS group_size
        FROM read_parquet('{INPUT_SQL_PATH}')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS duplicate_group_count,
        COALESCE(SUM(group_size), 0) AS rows_in_duplicate_groups,
        COALESCE(SUM(group_size - 1), 0) AS excess_duplicate_rows,
        COALESCE(MAX(group_size), 1) AS maximum_group_size
    FROM duplicate_groups
    """
).df()

total_row_count = connection.execute(
    f"SELECT COUNT(*) FROM read_parquet('{INPUT_SQL_PATH}')"
).fetchone()[0]

assert total_row_count == EXPECTED_ROW_COUNT

duplicate_summary["rows_in_duplicate_groups_share"] = (
    duplicate_summary["rows_in_duplicate_groups"] / total_row_count
)
duplicate_summary["excess_duplicate_rows_share"] = (
    duplicate_summary["excess_duplicate_rows"] / total_row_count
)

display(
    duplicate_summary.style.format(
        {
            "duplicate_group_count": "{:,.0f}",
            "rows_in_duplicate_groups": "{:,.0f}",
            "excess_duplicate_rows": "{:,.0f}",
            "maximum_group_size": "{:,.0f}",
            "rows_in_duplicate_groups_share": "{:.2%}",
            "excess_duplicate_rows_share": "{:.2%}",
        }
    )
)

,duplicate_group_count,rows_in_duplicate_groups,excess_duplicate_rows,maximum_group_size,rows_in_duplicate_groups_share,excess_duplicate_rows_share
0,"961,605","2,221,150","1,259,545",12,15.89%,9.01%


In [3]:
duplicate_group_count = int(duplicate_summary.loc[0, "duplicate_group_count"])
rows_in_duplicate_groups = int(duplicate_summary.loc[0, "rows_in_duplicate_groups"])
excess_duplicate_rows = int(duplicate_summary.loc[0, "excess_duplicate_rows"])
maximum_duplicate_group_size = int(duplicate_summary.loc[0, "maximum_group_size"])
excess_duplicate_share = float(duplicate_summary.loc[0, "excess_duplicate_rows_share"])

display(
    Markdown(
        f"""
### Practical Meaning

Exact duplicate records may represent repeated anonymized profiles rather than duplicated users. Removing them without a user identifier could incorrectly delete valid experimental observations.

### How to Interpret

The dataset contains **{duplicate_group_count:,}** exact duplicate groups involving **{rows_in_duplicate_groups:,}** rows. Relative to keeping one row per group, there are **{excess_duplicate_rows:,}** additional rows ({excess_duplicate_share:.2%} of the dataset), and the largest exact group contains **{maximum_duplicate_group_size}** rows.

### Final Conclusion

Exact duplicates are documented but retained. Their presence creates a potential train-test leakage risk, which is addressed by grouping identical pre-treatment feature vectors into the same partition.
"""
    )
)


### Practical Meaning

Exact duplicate records may represent repeated anonymized profiles rather than duplicated users. Removing them without a user identifier could incorrectly delete valid experimental observations.

### How to Interpret

The dataset contains **961,605** exact duplicate groups involving **2,221,150** rows. Relative to keeping one row per group, there are **1,259,545** additional rows (9.01% of the dataset), and the largest exact group contains **12** rows.

### Final Conclusion

Exact duplicates are documented but retained. Their presence creates a potential train-test leakage risk, which is addressed by grouping identical pre-treatment feature vectors into the same partition.


## Step 3 — Audit Repeated Pre-treatment Feature Vectors

This step checks whether multiple rows share the same `f0–f11` values, regardless of treatment assignment or outcome.

In [4]:
feature_duplicate_summary = connection.execute(
    f"""
    WITH feature_groups AS (
        SELECT
            {feature_sql},
            COUNT(*) AS group_size,
            COUNT(DISTINCT treatment) AS treatment_level_count
        FROM read_parquet('{INPUT_SQL_PATH}')
        GROUP BY {feature_sql}
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS repeated_feature_group_count,
        SUM(group_size) AS rows_in_repeated_feature_groups,
        SUM(group_size - 1) AS excess_rows_by_feature_vector,
        MAX(group_size) AS maximum_feature_group_size,
        COUNT_IF(treatment_level_count = 2) AS groups_with_both_treatments
    FROM feature_groups
    """
).df()

feature_duplicate_summary["rows_in_repeated_feature_groups_share"] = (
    feature_duplicate_summary["rows_in_repeated_feature_groups"]
    / total_row_count
)

display(
    feature_duplicate_summary.style.format(
        {
            "repeated_feature_group_count": "{:,.0f}",
            "rows_in_repeated_feature_groups": "{:,.0f}",
            "excess_rows_by_feature_vector": "{:,.0f}",
            "maximum_feature_group_size": "{:,.0f}",
            "groups_with_both_treatments": "{:,.0f}",
            "rows_in_repeated_feature_groups_share": "{:.2%}",
        }
    )
)

,repeated_feature_group_count,rows_in_repeated_feature_groups,excess_rows_by_feature_vector,maximum_feature_group_size,groups_with_both_treatments,rows_in_repeated_feature_groups_share
0,"1,185,455","2,811,714","1,626,259",13,"356,008",20.11%


## Step 4 — Define the Deterministic Partition Strategy

The partition is based only on a seeded MD5 signature of `f0–f11`:

- Buckets 0–59: `train`
- Buckets 60–79: `validation`
- Buckets 80–99: `test`

Treatment, exposure, visit, and conversion are excluded from the partition key. Identical feature vectors therefore remain in the same partition without using post-treatment information.

## Step 5 — Materialize the Partitioned Dataset

In [6]:
if TEMP_OUTPUT_FILE.exists():
    TEMP_OUTPUT_FILE.unlink()

start_time = time.perf_counter()

connection.execute(
    f"""
    COPY (
        SELECT
            *,
            {partition_expression} AS data_partition
        FROM read_parquet('{INPUT_SQL_PATH}')
    )
    TO '{TEMP_OUTPUT_SQL_PATH}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD,
        ROW_GROUP_SIZE 100000
    )
    """
)

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

TEMP_OUTPUT_FILE.replace(OUTPUT_FILE)

elapsed_seconds = time.perf_counter() - start_time
output_size_mb = OUTPUT_FILE.stat().st_size / (1024**2)

print(f"Partitioned dataset created in {elapsed_seconds:.1f} seconds.")
print(f"Output size: {output_size_mb:,.1f} MB")
print(f"Output path: {OUTPUT_FILE}")

Partitioned dataset created in 8.4 seconds.
Output size: 179.9 MB
Output path: /Users/ever/Library/CloudStorage/OneDrive2-ColumbiaUniversityIrvingMedicalCenter/adlift项目/data/processed/criteo-uplift-v2.1-partitioned.parquet


### Practical Meaning

The complete analysis-ready dataset now includes a reusable `data_partition` column while preserving every original row and variable.

### Final Conclusion

The deterministic partitioned Parquet file was created successfully with ZSTD compression. No source records were deleted.

## Step 6 — Validate Partition Sizes and Rates

Each partition is checked for row coverage, treatment allocation, exposure rate, visit rate, and conversion rate.

In [7]:
partition_summary = connection.execute(
    f"""
    SELECT
        data_partition,
        COUNT(*) AS user_count,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS user_share,
        AVG(treatment) AS treatment_rate,
        AVG(exposure) AS exposure_rate,
        AVG(visit) AS visit_rate,
        AVG(conversion) AS conversion_rate
    FROM read_parquet('{OUTPUT_SQL_PATH}')
    GROUP BY data_partition
    ORDER BY CASE data_partition
        WHEN 'train' THEN 1
        WHEN 'validation' THEN 2
        WHEN 'test' THEN 3
    END
    """
).df()

overall_rates = connection.execute(
    f"""
    SELECT
        AVG(treatment) AS treatment_rate,
        AVG(exposure) AS exposure_rate,
        AVG(visit) AS visit_rate,
        AVG(conversion) AS conversion_rate
    FROM read_parquet('{OUTPUT_SQL_PATH}')
    """
).df().iloc[0]

for rate_column in ["treatment_rate", "exposure_rate", "visit_rate", "conversion_rate"]:
    partition_summary[f"{rate_column}_delta"] = (
        partition_summary[rate_column] - overall_rates[rate_column]
    )

display(
    partition_summary.style.format(
        {
            "user_count": "{:,.0f}",
            "user_share": "{:.2%}",
            "treatment_rate": "{:.4%}",
            "exposure_rate": "{:.4%}",
            "visit_rate": "{:.4%}",
            "conversion_rate": "{:.4%}",
            "treatment_rate_delta": "{:+.4%}",
            "exposure_rate_delta": "{:+.4%}",
            "visit_rate_delta": "{:+.4%}",
            "conversion_rate_delta": "{:+.4%}",
        }
    )
)

,data_partition,user_count,user_share,treatment_rate,exposure_rate,visit_rate,conversion_rate,treatment_rate_delta,exposure_rate_delta,visit_rate_delta,conversion_rate_delta
0,train,"8,389,214",60.01%,84.9937%,3.0642%,4.6985%,0.2919%,-0.0063%,+0.0011%,-0.0007%,+0.0002%
1,validation,"2,794,629",19.99%,85.0138%,3.0640%,4.6924%,0.2900%,+0.0138%,+0.0009%,-0.0068%,-0.0017%
2,test,"2,795,749",20.00%,85.0052%,3.0590%,4.7083%,0.2927%,+0.0052%,-0.0041%,+0.0091%,+0.0011%


In [8]:
expected_partitions = {"train", "validation", "test"}
observed_partitions = set(partition_summary["data_partition"])

assert observed_partitions == expected_partitions
assert int(partition_summary["user_count"].sum()) == EXPECTED_ROW_COUNT
assert partition_summary["data_partition"].notna().all()

expected_shares = {"train": 0.60, "validation": 0.20, "test": 0.20}

for _, row in partition_summary.iterrows():
    expected_share = expected_shares[row["data_partition"]]
    assert abs(row["user_share"] - expected_share) < 0.01
    assert abs(row["treatment_rate_delta"]) < 0.002

print("Partition size and treatment-rate validation passed.")

Partition size and treatment-rate validation passed.


In [9]:
partition_counts = dict(
    zip(partition_summary["data_partition"], partition_summary["user_count"])
)
partition_shares = dict(
    zip(partition_summary["data_partition"], partition_summary["user_share"])
)
maximum_treatment_delta = partition_summary["treatment_rate_delta"].abs().max()
maximum_visit_delta = partition_summary["visit_rate_delta"].abs().max()
maximum_conversion_delta = partition_summary["conversion_rate_delta"].abs().max()

display(
    Markdown(
        f"""
### Practical Meaning

Balanced partition sizes and rates ensure that model training, tuning, and final evaluation use comparable experimental populations.

### How to Interpret

The output contains **{int(partition_counts['train']):,}** train rows ({partition_shares['train']:.2%}), **{int(partition_counts['validation']):,}** validation rows ({partition_shares['validation']:.2%}), and **{int(partition_counts['test']):,}** test rows ({partition_shares['test']:.2%}). The largest absolute partition deviation from the overall treatment rate is {maximum_treatment_delta:.4%}; visit-rate deviation is {maximum_visit_delta:.4%}; and conversion-rate deviation is {maximum_conversion_delta:.4%}.

### Final Conclusion

The partition sizes match the intended 60/20/20 design, and treatment and outcome rates remain stable across partitions.
"""
    )
)


### Practical Meaning

Balanced partition sizes and rates ensure that model training, tuning, and final evaluation use comparable experimental populations.

### How to Interpret

The output contains **8,389,214** train rows (60.01%), **2,794,629** validation rows (19.99%), and **2,795,749** test rows (20.00%). The largest absolute partition deviation from the overall treatment rate is 0.0138%; visit-rate deviation is 0.0091%; and conversion-rate deviation is 0.0017%.

### Final Conclusion

The partition sizes match the intended 60/20/20 design, and treatment and outcome rates remain stable across partitions.


## Step 7 — Verify Reproducibility and Feature-group Isolation

This step confirms that every stored partition matches the deterministic rule and that no identical pre-treatment feature vector appears in multiple partitions.

In [10]:
partition_rule_mismatches = connection.execute(
    f"""
    SELECT COUNT_IF(data_partition <> ({partition_expression}))
    FROM read_parquet('{OUTPUT_SQL_PATH}')
    """
).fetchone()[0]

cross_partition_feature_groups = connection.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT {feature_sql}
        FROM read_parquet('{OUTPUT_SQL_PATH}')
        GROUP BY {feature_sql}
        HAVING COUNT(DISTINCT data_partition) > 1
    )
    """
).fetchone()[0]

assert partition_rule_mismatches == 0
assert cross_partition_feature_groups == 0

reproducibility_checks = pd.DataFrame(
    {
        "check": [
            "Partition-rule mismatches",
            "Feature groups crossing partitions",
        ],
        "failure_count": [
            partition_rule_mismatches,
            cross_partition_feature_groups,
        ],
    }
)

display(reproducibility_checks)

,check,failure_count
0,Partition-rule mismatches,0
1,Feature groups crossing partitions,0


### Practical Meaning

These checks protect the final test set from feature-group leakage and confirm that partition labels can be reproduced from the documented rule.

### How to Interpret

A failure count of zero is required for both checks. Any nonzero value would mean that the saved dataset does not follow the documented partition rule or that identical feature vectors leak across partitions.

### Final Conclusion

The partition rule has zero mismatches, and zero identical feature groups cross partition boundaries. The split is reproducible and leakage-aware.

## Step 8 — Phase 1 Readiness Summary

In [11]:
display(
    Markdown(
        f"""
## Phase 1 Final Conclusion

The full **{EXPECTED_ROW_COUNT:,}-row** Criteo dataset has passed schema, completeness, binary-field, logical-consistency, treatment-control balance, duplicate, and partition validation.

Exact duplicate records were retained because the dataset does not provide a user identifier. A deterministic feature-group hash split prevents identical pre-treatment profiles from crossing train, validation, and test boundaries.

The partitioned Parquet dataset is analysis-ready for uplift-model development and out-of-sample policy evaluation.
"""
    )
)

connection.close()
print("Database connection closed.")


## Phase 1 Final Conclusion

The full **13,979,592-row** Criteo dataset has passed schema, completeness, binary-field, logical-consistency, treatment-control balance, duplicate, and partition validation.

Exact duplicate records were retained because the dataset does not provide a user identifier. A deterministic feature-group hash split prevents identical pre-treatment profiles from crossing train, validation, and test boundaries.

The partitioned Parquet dataset is analysis-ready for uplift-model development and out-of-sample policy evaluation.


Database connection closed.
